# Peek at Pfam GFF — annotation rows ↔ biosamples / studies

Worked-example queries showing how to navigate from a row in
`nmdc_results.pfam_annotation_gff` out to the biosample, study, and
file metadata in `nmdc_metadata`. Mirrors `peek_ko_ec_links.ipynb`.

**Anchor columns on `pfam_annotation_gff`:**

| column | meaning | join target |
|---|---|---|
| `workflow_run_id` | `nmdc:wfmgan-...` MetagenomeAnnotation activity | `nmdc_metadata.workflow_execution_set.id` |
| `data_object_id`  | `nmdc:dobj-...` source GFF file | `nmdc_metadata.data_object_set.id` |
| `gene_id`         | `<workflow_run_id>_<contig>_<start>_<end>` | local; matches functional GFFs (issues #84, #86–#90) |
| `pfam_accession`  | `PFxxxxx` (e.g., `PF00005`) | Pfam ontology / `pfam_terms` reference |

**Pfam-specific hit-quality columns** (not present on KO/EC tables):

| column | meaning |
|---|---|
| `score`            | HMMER bit score |
| `e_value`          | HMMER e-value |
| `alignment_length` | aligned residues |
| `model_start`, `model_end` | HMM coords aligned to the gene |

**The chain (identical to KO/EC):**

```
pfam_annotation_gff.workflow_run_id
  → workflow_execution_set.id  (.was_informed_by is array of dgns IDs)
  → data_generation_set.id
    → data_generation_set_has_input.has_input            (biosample IDs)
    → data_generation_set_associated_studies.associated_studies
  → biosample_set.{env_*, geo_loc_name_*, depth, …}
```


### Session setup

In [1]:
spark = get_spark_session(app_name="peek_pfam_gff", tenant_name="nmdc")
print(f"Spark version: {spark.version}")

Spark version: 4.0.1


### Preflight: confirm the Pfam table is registered

In [2]:
registered = {r.tableName for r in spark.sql("SHOW TABLES IN nmdc_results").collect()}
if "pfam_annotation_gff" not in registered:
    raise RuntimeError("nmdc_results.pfam_annotation_gff not registered — run ingest_pfam_gff.ipynb first")
print("OK: nmdc_results.pfam_annotation_gff")

OK: nmdc_results.pfam_annotation_gff


### 1. Pick an anchor row

Grab one Pfam hit to drive the rest of the joins.

In [3]:
_anchor_df = spark.sql("""
    SELECT gene_id, pfam_accession, workflow_run_id, data_object_id,
           score, e_value, alignment_length
    FROM   nmdc_results.pfam_annotation_gff
    WHERE  workflow_run_id IS NOT NULL
      AND  data_object_id  IS NOT NULL
      AND  pfam_accession  IS NOT NULL
    LIMIT 1
""").toPandas()
if _anchor_df.empty:
    raise RuntimeError("no row in nmdc_results.pfam_annotation_gff has non-null workflow_run_id/data_object_id/pfam_accession")
anchor = _anchor_df.iloc[0]
WORKFLOW_RUN_ID = anchor["workflow_run_id"]
DATA_OBJECT_ID  = anchor["data_object_id"]
PFAM_ACCESSION  = anchor["pfam_accession"]
print(anchor.to_string())

Kernel token updated from session file


gene_id             nmdc:wfmgan-11-0b699133.1_00001_1462_2169
pfam_accession                                        PF13358
workflow_run_id                     nmdc:wfmgan-11-0b699133.1
data_object_id                          nmdc:dobj-11-2gt95549
score                                                    98.4
e_value                                                   0.0
alignment_length                                          145


### 2. `workflow_run_id` → workflow_execution_set

`was_informed_by` is array-typed on `workflow_execution_set` (a workflow run
can be informed by multiple data generations). The Silver layer flattens
this into `nmdc_metadata.workflow_execution_set_was_informed_by` (one row
per `(parent_id, was_informed_by)` pair) — prefer this side table over
`EXPLODE` for downstream joins.

In [4]:
spark.sql(f"""
    SELECT id, type, was_informed_by, started_at_time, ended_at_time
    FROM nmdc_metadata.workflow_execution_set
    WHERE id = '{WORKFLOW_RUN_ID}'
""").toPandas()

,id,type,was_informed_by,started_at_time,ended_at_time
0,nmdc:wfmgan-11-0b699133.1,nmdc:MetagenomeAnnotation,[nmdc:dgns-11-94abb207],2025-05-06T21:21:10.253394+00:00,NaN


### 3. workflow_execution → data_generation_set (the sequencing run)

In [5]:
spark.sql(f"""
    SELECT dg.id, dg.type, dg.name, dg.processing_institution,
           dg.principal_investigator_name, dg.start_date, dg.end_date
    FROM   nmdc_metadata.workflow_execution_set_was_informed_by wib
    JOIN   nmdc_metadata.data_generation_set dg
             ON dg.id = wib.was_informed_by
    WHERE  wib.parent_id = '{WORKFLOW_RUN_ID}'
""").toPandas()

,id,type,name,processing_institution,principal_investigator_name,start_date,end_date
0,nmdc:dgns-11-94abb207,nmdc:NucleotideSequencing,Terrestrial soil microbial communities - DEJU_...,Battelle,NaN,NaN,NaN


### Aside: provenance graph and `biosample_to_workflow_run`

NMDC's data model has two parallel chains, bridged by `data_generation`:

```
biosample → material_processing → procsm → … → data_generation
                                                       ↓
                              data_object ← workflow_execution
```

Walking this graph from a workflow run back to source biosamples was previously
done with a Trino `WITH RECURSIVE` CTE (which hits a 150-stage planner limit
at scale) or an iterative Python BFS. Both are now replaced by a single JOIN
to the precomputed `nmdc_metadata.biosample_to_workflow_run` table.

That table was built by `notebooks/build_biosample_to_workflow_run.ipynb`
and must be rebuilt after each NMDC data load.

### 4. workflow_execution → biosample(s)

A direct JOIN to `biosample_to_workflow_run` replaces the recursive graph
walk. The precomputed table covers any chain depth and records which
MaterialProcessing classes appeared in the provenance chain.

In [6]:
spark.sql(f"""
    SELECT b2wr.biosample_id, b2wr.n_hops,
           bsm.env_broad_scale_term_id, bsm.env_local_scale_term_id,
           bsm.env_medium_term_id, bsm.geo_loc_name_has_raw_value,
           bsm.depth_has_numeric_value, bsm.depth_has_unit
    FROM   nmdc_metadata.biosample_to_workflow_run b2wr
    JOIN   nmdc_metadata.biosample_set bsm ON bsm.id = b2wr.biosample_id
    WHERE  b2wr.workflow_run_id = '{WORKFLOW_RUN_ID}'
""").toPandas()

,biosample_id,n_hops,env_broad_scale_term_id,env_local_scale_term_id,env_medium_term_id,geo_loc_name_has_raw_value,depth_has_numeric_value,depth_has_unit
0,nmdc:bsm-11-k33xda67,8,ENVO:00000446,ENVO:01000843,ENVO:00001998,"USA: Alaska, Delta Junction",NaN,m
1,nmdc:bsm-11-pm9paj46,8,ENVO:00000446,ENVO:01000843,ENVO:00001998,"USA: Alaska, Delta Junction",NaN,m
2,nmdc:bsm-11-rb7txp53,8,ENVO:00000446,ENVO:01000843,ENVO:00001998,"USA: Alaska, Delta Junction",NaN,m


### 5. data_generation → study

In [7]:
spark.sql(f"""
    SELECT s.id AS study_id, s.name, s.title, s.principal_investigator_name
    FROM   nmdc_metadata.workflow_execution_set_was_informed_by wib
    JOIN   nmdc_metadata.data_generation_set_associated_studies dgs
             ON dgs.parent_id = wib.was_informed_by
    JOIN   nmdc_metadata.study_set s
             ON s.id = dgs.associated_studies
    WHERE  wib.parent_id = '{WORKFLOW_RUN_ID}'
""").toPandas()

,study_id,name,title,principal_investigator_name
0,nmdc:sty-11-34xj1150,National Ecological Observatory Network: soil ...,National Ecological Observatory Network: soil ...,Kate Thibault


### 6. `data_object_id` → data_object_set (file-level provenance)

Useful for verifying which source GFF a row came from, file size, MD5,
URL, etc. Not needed for downstream science.

In [8]:
spark.sql(f"""
    SELECT id, name, data_object_type, file_size_bytes, md5_checksum, url
    FROM nmdc_metadata.data_object_set
    WHERE id = '{DATA_OBJECT_ID}'
""").toPandas()

,id,name,data_object_type,file_size_bytes,md5_checksum,url
0,nmdc:dobj-11-2gt95549,nmdc_wfmgan-11-0b699133.1_pfam.gff,Pfam Annotation GFF,3036662,3dd7a5b12446b0d7ee4d4ad7f60fc002,https://data.microbiomedata.org/data/nmdc:dgns...


### 7. Cross-check against `functional_annotation_agg`

`functional_annotation_agg` is a precomputed `(workflow_run, gene_function_id, count)`
rollup that already carries PFAM (along with KEGG.ORTHOLOGY and COG; EC is
absent — see `peek_ko_ec_links` for that wrinkle). Two caveats when comparing
to `pfam_annotation_gff`:

1. **Prefix differs** — `pfam_accession = 'PFxxxxx'` here vs
   `gene_function_id = 'PFAM:PFxxxxx'` there. Translate before joining.
2. **Multiple HMM hits per gene** — the GFF can record more than one Pfam hit
   per gene (different model coords / overlapping families). The agg may have
   been deduplicated to one row per `(workflow_run, gene, pfam)`. Drift in this
   direction is expected; raw `COUNT(*)` in `pfam_annotation_gff` ≥ `SUM(count)`
   in the agg per workflow run is the typical pattern. Larger drift is worth flagging.

In [9]:
spark.sql(f"""
    WITH ours AS (
        SELECT 'PFAM:' || pfam_accession AS gene_function_id,
               COUNT(*) AS hits_in_pfam_table
        FROM nmdc_results.pfam_annotation_gff
        WHERE workflow_run_id = '{WORKFLOW_RUN_ID}'
        GROUP BY 1
    ),
    theirs AS (
        SELECT gene_function_id, count AS count_in_agg
        FROM nmdc_metadata.functional_annotation_agg
        WHERE was_generated_by = '{WORKFLOW_RUN_ID}'
          AND gene_function_id LIKE 'PFAM:%'
    )
    SELECT COALESCE(ours.gene_function_id, theirs.gene_function_id) AS gene_function_id,
           ours.hits_in_pfam_table,
           theirs.count_in_agg,
           COALESCE(ours.hits_in_pfam_table, 0) - COALESCE(theirs.count_in_agg, 0) AS diff
    FROM ours FULL OUTER JOIN theirs USING (gene_function_id)
    ORDER BY ABS(COALESCE(ours.hits_in_pfam_table, 0) - COALESCE(theirs.count_in_agg, 0)) DESC
    LIMIT 20
""").toPandas()

,gene_function_id,hits_in_pfam_table,count_in_agg,diff
0,PFAM:PF07676,41,26,15
1,PFAM:PF07494,22,12,10
2,PFAM:PF08238,15,6,9
3,PFAM:PF00575,27,18,9
4,PFAM:PF03989,15,8,7
5,PFAM:PF02321,72,67,5
6,PFAM:PF13517,31,26,5
7,PFAM:PF01396,7,3,4
8,PFAM:PF00623,9,6,3
9,PFAM:PF03707,10,7,3


### 8. End-to-end: Pfam hits per biosample for a chosen accession

A single Spark SQL query joins `pfam_annotation_gff` to
`biosample_to_workflow_run` — no recursive walk, no Trino cursor.
The precomputed provenance bridge does the graph traversal for free.

Filter on `pfam_accession` (a low-cardinality column) gets pushed down to
parquet column statistics, so this should be fast even though the Silver
table is ~50 GiB.

In [11]:
import time
TARGET_PFAM = PFAM_ACCESSION  # substitute any 'PFxxxxx', e.g. 'PF00005' (ABC transporter)

t0 = time.monotonic()
result = spark.sql(f"""
    SELECT b2wr.biosample_id,
           bsm.env_broad_scale_term_id,
           bsm.env_medium_term_id,
           bsm.geo_loc_name_has_raw_value,
           bsm.depth_has_numeric_value,
           SUM(p.n_hits) AS total_hits
    FROM (
        SELECT workflow_run_id, COUNT(*) AS n_hits
        FROM   nmdc_results.pfam_annotation_gff
        WHERE  pfam_accession = '{TARGET_PFAM}'
        GROUP  BY workflow_run_id
    ) p
    JOIN nmdc_metadata.biosample_to_workflow_run b2wr
      ON b2wr.workflow_run_id = p.workflow_run_id
    JOIN nmdc_metadata.biosample_set bsm
      ON bsm.id = b2wr.biosample_id
    GROUP BY b2wr.biosample_id, bsm.env_broad_scale_term_id,
             bsm.env_medium_term_id, bsm.geo_loc_name_has_raw_value,
             bsm.depth_has_numeric_value
    ORDER BY total_hits DESC
    LIMIT 20
""").toPandas()
print(f"{len(result)} biosamples  ({time.monotonic()-t0:.1f}s)")
result

20 biosamples  (22.2s)


,biosample_id,env_broad_scale_term_id,env_medium_term_id,geo_loc_name_has_raw_value,depth_has_numeric_value,total_hits
0,nmdc:bsm-11-grdf8n16,ENVO:00000446,ENVO:00001998,"USA: Utah, Onaqui",NaN,8046
1,nmdc:bsm-11-qvm3bq92,ENVO:00000446,ENVO:00001998,"USA: Utah, Onaqui",NaN,8046
2,nmdc:bsm-11-3rbwrr59,ENVO:00000446,ENVO:00001998,"USA: Utah, Onaqui",NaN,8046
3,nmdc:bsm-11-y7nhh683,ENVO:01000219,ENVO:00002259,"USA: Washington, Prosser Non-irrigated Bare",NaN,6779
4,nmdc:bsm-11-4vfwry15,ENVO:00000446,ENVO:00001998,"USA: Alabama, Talladega National Forest",NaN,5963
5,nmdc:bsm-11-ecjza388,ENVO:00000446,ENVO:00001998,"USA: Alabama, Talladega National Forest",NaN,5963
6,nmdc:bsm-11-t1s9v252,ENVO:00000446,ENVO:00001998,"USA: Alabama, Talladega National Forest",NaN,5963
7,nmdc:bsm-11-4ttw3s39,ENVO:00000446,ENVO:00001998,"USA: Wisconsin, Steigerwaldt-Chequamegon",NaN,5049
8,nmdc:bsm-11-1eqbdk44,ENVO:00000446,ENVO:00001998,"USA: Wisconsin, Steigerwaldt-Chequamegon",NaN,5049
9,nmdc:bsm-11-4ga25m92,ENVO:00000446,ENVO:00001998,"USA: Wisconsin, Steigerwaldt-Chequamegon",NaN,5049


### 9. Pfam-only columns: hit-quality summary

The columns `score`, `e_value`, `alignment_length`, `model_start`, `model_end`
are unique to `pfam_annotation_gff` (the KO/EC tables don't carry HMM-level
detail). Top accessions for the anchor workflow run, with quality stats:

In [12]:
spark.sql(f"""
    SELECT pfam_accession,
           COUNT(*) AS n_hits,
           ROUND(AVG(score), 1)            AS avg_bit_score,
           MIN(e_value)                    AS min_e_value,
           MAX(e_value)                    AS max_e_value,
           ROUND(AVG(alignment_length), 1) AS avg_align_len
    FROM   nmdc_results.pfam_annotation_gff
    WHERE  workflow_run_id = '{WORKFLOW_RUN_ID}'
    GROUP  BY pfam_accession
    ORDER  BY n_hits DESC
    LIMIT  20
""").toPandas()

,pfam_accession,n_hits,avg_bit_score,min_e_value,max_e_value,avg_align_len
0,PF00873,223,87.7,1.500000e-89,2.700000e-07,114.3
1,PF00005,202,47.5,1.700000e-29,1.800000e-05,76.7
2,PF00106,116,71.9,9.500000e-47,1.700000e-06,98.5
3,PF07690,100,58.9,3.400000e-39,2.200000e-08,122.2
4,PF00072,93,61.2,1.300000e-30,6.100000e-08,85.7
5,PF00171,78,102.1,1.300000e-81,3.400000e-07,106.7
6,PF12704,76,44.2,4.300000e-29,4.500000e-06,125.9
7,PF00501,75,63.1,3.300000e-53,1.500000e-07,107.0
8,PF13561,75,79.5,6.300000e-57,1.900000e-06,99.9
9,PF02321,72,42.9,1.200000e-25,1.000000e-04,104.7


### 10. Annotation run QC — `annotation_statistics`

`nmdc_results.annotation_statistics` has one row per annotation workflow run with 17 QC
metrics: sequence counts, gene-type counts (CDS, tRNA, ncRNA, rRNA, CRISPR), coding
density, and genes-per-Mbp. Same join shape as for KO/EC.

In [13]:
if "annotation_statistics" not in {r.tableName for r in spark.sql("SHOW TABLES IN nmdc_results").collect()}:
    print("SKIP: annotation_statistics not registered")
else:
    display(spark.sql(f"""
        SELECT ann.workflow_run_id,
               ann.n_seqs, ann.n_cds, ann.n_trna, ann.n_rrna,
               ann.coding_density_pct, ann.genes_per_1m_bp,
               bsm.id AS biosample_id, bsm.env_broad_scale_term_id
        FROM   nmdc_results.annotation_statistics ann
        JOIN   nmdc_metadata.biosample_to_workflow_run b2wr
                 ON b2wr.workflow_run_id = ann.workflow_run_id
        JOIN   nmdc_metadata.biosample_set bsm
                 ON bsm.id = b2wr.biosample_id
        WHERE  ann.workflow_run_id = '{WORKFLOW_RUN_ID}'
    """).toPandas())

,workflow_run_id,n_seqs,n_cds,n_trna,n_rrna,coding_density_pct,genes_per_1m_bp,biosample_id,env_broad_scale_term_id
0,nmdc:wfmgan-11-0b699133.1,24608,27759,152.0,134.0,89.07,3128.68,nmdc:bsm-11-k33xda67,ENVO:00000446
1,nmdc:wfmgan-11-0b699133.1,24608,27759,152.0,134.0,89.07,3128.68,nmdc:bsm-11-pm9paj46,ENVO:00000446
2,nmdc:wfmgan-11-0b699133.1,24608,27759,152.0,134.0,89.07,3128.68,nmdc:bsm-11-rb7txp53,ENVO:00000446


#### Distribution across biosamples

Coding density and gene content across all annotation runs, grouped by broad-scale environment.

In [14]:
if "annotation_statistics" not in {r.tableName for r in spark.sql("SHOW TABLES IN nmdc_results").collect()}:
    print("SKIP: annotation_statistics not registered")
else:
    display(spark.sql("""
        SELECT bsm.env_broad_scale_term_id,
               COUNT(DISTINCT b2wr.biosample_id)  AS n_biosamples,
               ROUND(AVG(ann.coding_density_pct), 2) AS avg_coding_density,
               ROUND(AVG(ann.genes_per_1m_bp), 0)    AS avg_genes_per_mbp,
               ROUND(AVG(ann.n_cds), 0)               AS avg_n_cds
        FROM   nmdc_results.annotation_statistics ann
        JOIN   nmdc_metadata.biosample_to_workflow_run b2wr
                 ON b2wr.workflow_run_id = ann.workflow_run_id
        JOIN   nmdc_metadata.biosample_set bsm
                 ON bsm.id = b2wr.biosample_id
        GROUP BY bsm.env_broad_scale_term_id
        ORDER BY n_biosamples DESC
        LIMIT 20
    """).toPandas())

,env_broad_scale_term_id,n_biosamples,avg_coding_density,avg_genes_per_mbp,avg_n_cds
0,ENVO:00000446,3330,88.39,2887.0,462780.0
1,ENVO:01000253,725,85.96,2673.0,324393.0
2,ENVO:01000252,500,86.99,2114.0,2337302.0
3,ENVO:03605008,74,86.75,3070.0,69289.0
4,ENVO:01000174,45,88.02,2875.0,3813663.0
5,ENVO:01000219,40,83.85,1695.0,1436969.0
6,ENVO:01000215,4,91.52,2567.0,5637356.0
7,ENVO:01000216,3,90.22,2319.0,8858437.0
8,ENVO:01000247,2,91.30,2213.0,7419847.0
9,ENVO:01000221,2,88.81,2479.0,4846895.0


Kernel token updated from session file
Kernel token updated from session file
